# ゲーム売上予測モデル（vgchartz-2024）

世界のゲーム販売データ（vgchartz-2024）を用いて、各タイトルの総売上 `total_sales`（百万本）を予測する回帰モデルを構築する。
ベースラインの線形回帰から始め、LightGBM・データ期間の限定・テキスト特徴量（TF-IDF）・Target Encoding と段階的に改善し、最終的に **テスト R² ≈ 0.65** を達成した。

**改善の流れ**
1. 線形回帰（One-Hot）でベースラインを確認 → 当てはまりが悪い
2. LightGBM ＋ 目的変数の対数変換
3. 特徴量追加（タイトル長・交互作用 など）
4. EDA を見直し、バイアスの大きい 2000 年以前を除外
5. タイトルの TF-IDF ＋ Target Encoding ＋ ハイパーパラメータ調整

## 0. ライブラリ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. データ読み込み

vgchartz-2024 の CSV を読み込む。Colab では `/content/` 配下、ローカルでは `data/` 配下などに配置する。

In [ ]:
df_game = pd.read_csv("vgchartz-2024.csv")
df_game

## 2. 探索的データ分析（EDA）

まず全体像と目的変数 `total_sales` の分布、欠損状況を確認する。

In [ ]:
df_game.describe()

In [ ]:
# 目的変数の分布（裾が長いので 0〜5 に拡大して観察）
df_game["total_sales"].hist(bins=90)
plt.xlim(0, 5)
plt.show()

df_game.boxplot(column="total_sales")
plt.show()

In [ ]:
# 欠損値の確認
df_game.isnull().sum()

## 3. 目的変数の欠損を除去

`total_sales` が欠損している行は学習・評価に使えないため除外する。
残ったデータについて、各列の欠損率とユニーク数を一覧化する。

In [ ]:
df_model = df_game[df_game["total_sales"].notna()].copy()

summary = pd.DataFrame({
    "missing_count": df_model.isnull().sum(),
    "missing_rate": df_model.isnull().mean() * 100,
    "nunique": df_model.nunique(),
})
summary.sort_values("missing_rate", ascending=False)

## 4. 特徴量エンジニアリング（発売日）

`release_date` を datetime 化し、発売年・発売月を抽出。さらに 10 年単位の `decade` を作る。

In [ ]:
df_model["release_date"] = pd.to_datetime(df_model["release_date"])
df_model["release_year"] = df_model["release_date"].dt.year
df_model["release_month"] = df_model["release_date"].dt.month

# 10 年単位
df_model["decade"] = (df_model["release_year"] // 10) * 10

In [ ]:
# 発売年ごとの件数・平均・中央値
df_model.groupby("release_year")["total_sales"].agg(
    count="count", mean="mean", median="median"
)

In [ ]:
# 発売年ごとのタイトル数
year_count = df_model.groupby("release_year").size()

plt.figure(figsize=(12, 5))
year_count.plot(kind="bar")
plt.xlabel("Release Year")
plt.ylabel("Count")
plt.title("Number of Games by Release Year")
plt.show()

## 5. カテゴリ変数の集計

`developer` / `publisher` / `genre` / `console` と売上の関係を確認する。
件数の少ないカテゴリは後段で `Other` に集約する方針とする。

In [ ]:
# ジャンル別の売上
df_model.groupby("genre")["total_sales"].agg(
    count="count", mean="mean", median="median"
)

In [ ]:
# コンソール別の売上（平均が高い順）
console_stats = (
    df_model.groupby("console")["total_sales"]
    .agg(count="count", mean="mean", median="median")
    .sort_values("mean", ascending=False)
)
console_stats

In [ ]:
# 件数 100 以上の publisher / developer のみ残し、それ以外は Other に集約
for col in ["publisher", "developer"]:
    counts = df_model[col].value_counts()
    major = counts[counts >= 100].index
    df_model[f"{col}_group"] = df_model[col].where(df_model[col].isin(major), "Other")

df_model["publisher_group"].value_counts()

## 6. ベースライン：One-Hot Encoding ＋ 線形回帰

まずは単純な線形回帰で当てはまり具合を確認する。
過学習の有無を見るため、学習データとテストデータに分割して評価する。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

df_enc = df_model[
    ["total_sales", "console", "genre", "publisher_group", "decade"]
].dropna()

df_encoded = pd.get_dummies(
    df_enc,
    columns=["console", "genre", "publisher_group", "decade"],
    drop_first=True,
)

X = df_encoded.drop(columns=["total_sales"])
y = df_encoded["total_sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)

print("決定係数:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))

線形回帰では当てはまりが悪い（決定係数が低い）。`critic_score` との相関も弱く、線形の仮定では売上をうまく説明できない。
そこで非線形を捉えられる **LightGBM** に切り替え、裾の長い目的変数は対数変換（`log1p`）して扱う。

## 7. LightGBM（目的変数を対数変換）

`total_sales` は右に大きく裾を引くため `np.log1p` で変換し、勾配ブースティングで学習する。
`critic_score` の欠損は `-1` で埋め、欠損自体を情報として残す。

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

df_lgb = df_model.copy()
df_lgb["critic_score"] = df_lgb["critic_score"].fillna(-1)

# タイトル由来の特徴量
df_lgb["title_len"] = df_lgb["title"].str.len()
df_lgb["has_number"] = df_lgb["title"].str.contains(r"\d", na=False).astype(int)
df_lgb["title_freq"] = df_lgb["title"].map(df_lgb["title"].value_counts())

# 交互作用特徴量
df_lgb["console_genre"] = df_lgb["console"] + "_" + df_lgb["genre"]
df_lgb["publisher_genre"] = df_lgb["publisher_group"] + "_" + df_lgb["genre"]

features = [
    "console", "genre", "publisher_group", "developer_group",
    "console_genre", "publisher_genre",
    "release_year", "release_month", "decade",
    "critic_score", "title_len", "has_number", "title_freq",
]

X = pd.get_dummies(df_lgb[features], drop_first=True)
y = np.log1p(df_lgb["total_sales"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LGBMRegressor(
    n_estimators=1000, learning_rate=0.03,
    max_depth=6, num_leaves=31, random_state=42, verbose=-1,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("R²:", r2_score(y_test, pred))

## 8. EDA を見直し：2000 年以前を除外

発売年ごとの平均・中央値・件数を重ねて見ると、**1990 年以前はサンプルが少ないのに平均・中央値が大きい**。
これは「古い年代はヒット作だけがデータとして残っている」生存バイアスの可能性が高い。
学習を歪めるため、ここでは **2000 年以降**に限定する。

In [ ]:
year_stats = df_model.groupby("release_year")["total_sales"].agg(
    count="count", mean="mean", median="median", max="max"
)

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(year_stats.index, year_stats["mean"])
ax1.set_ylabel("Mean Sales")
ax2 = ax1.twinx()
ax2.bar(year_stats.index, year_stats["count"], alpha=0.3)
ax2.set_ylabel("Count")
plt.show()

In [ ]:
# 2000 年以降に限定
df_model = df_model[df_model["release_year"] >= 2000].copy()

# 年あたりの発売本数（その年の市場規模の代理指標）
year_amount = df_model.groupby("release_year").size()
df_model["year_amount"] = df_model["release_year"].map(year_amount)

# 件数 50 以上の publisher / developer のみ残す（しきい値を見直し）
for col in ["publisher", "developer"]:
    counts = df_model[col].value_counts()
    major = counts[counts >= 50].index
    df_model[f"{col}_group"] = df_model[col].where(df_model[col].isin(major), "Other")

## 9. LightGBM 再学習（year_amount 追加）

2000 年以降への限定と `year_amount` の追加で、テスト R² が約 0.3 改善した。

In [ ]:
df_lgb = df_model.copy()
df_lgb["critic_score"] = df_lgb["critic_score"].fillna(-1)

df_lgb["title_len"] = df_lgb["title"].str.len()
df_lgb["has_number"] = df_lgb["title"].str.contains(r"\d", na=False).astype(int)
df_lgb["title_freq"] = df_lgb["title"].map(df_lgb["title"].value_counts())
df_lgb["console_genre"] = df_lgb["console"] + "_" + df_lgb["genre"]
df_lgb["publisher_genre"] = df_lgb["publisher_group"] + "_" + df_lgb["genre"]

features = [
    "console", "genre", "publisher_group", "developer_group",
    "console_genre", "publisher_genre",
    "release_year", "release_month", "decade", "year_amount",
    "critic_score", "title_len", "has_number", "title_freq",
]

X = pd.get_dummies(df_lgb[features], drop_first=True)
y = np.log1p(df_lgb["total_sales"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LGBMRegressor(
    n_estimators=1000, learning_rate=0.03,
    max_depth=6, num_leaves=31, random_state=42, verbose=-1,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("R²:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

importance = pd.DataFrame({
    "feature": X.columns, "importance": model.feature_importances_
}).sort_values("importance", ascending=False)
importance.head(30)

## 10. タイトルのテキスト特徴量（TF-IDF）

タイトル文字列そのものにも情報があると考え、テキスト特徴量を追加する。

> **メモ（TF-IDF）**：単語を特徴量化する素朴な方法は Bag-of-Words（全単語を one-hot 化）だが、これは頻出語まで重視してしまう。
> TF-IDF は珍しい単語ほど重みを上げることで、識別力の高い語を特徴量に反映できる。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack

tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), stop_words="english")
X_title = tfidf.fit_transform(df_lgb["title"].fillna(""))

# 既存の数値・ダミー特徴量（疎行列）とタイトルTF-IDFを結合
X_sparse = csr_matrix(X.astype(np.float32))
X_combined = hstack([X_sparse, X_title])
print(X_sparse.shape, X_title.shape, X_combined.shape)

## 11. 最終モデル：Target Encoding ＋ TF-IDF ＋ チューニング

高カーディナリティのカテゴリ（`publisher` / `developer` など）を One-Hot ではなく **Target Encoding** で扱い、
タイトルの TF-IDF と結合して LightGBM をチューニングする。
Target Encoding はリークを避けるため、**学習データで `fit` → テストへ `transform`** の順で適用する。

In [ ]:
!pip install category_encoders -q

In [ ]:
from category_encoders import TargetEncoder

df_lgb = df_model.copy()
df_lgb["critic_score"] = df_lgb["critic_score"].fillna(-1)

year_count = df_lgb["release_year"].value_counts()
df_lgb["year_amount"] = df_lgb["release_year"].map(year_count)
df_lgb["title_len"] = df_lgb["title"].str.len()
df_lgb["has_number"] = df_lgb["title"].str.contains(r"\d", na=False).astype(int)
df_lgb["title_freq"] = df_lgb["title"].map(df_lgb["title"].value_counts())

df_lgb["console_genre"] = df_lgb["console"] + "_" + df_lgb["genre"]
df_lgb["publisher_genre"] = df_lgb["publisher"] + "_" + df_lgb["genre"]
df_lgb["developer_genre"] = df_lgb["developer"] + "_" + df_lgb["genre"]

cat_cols = [
    "console", "genre", "publisher", "developer",
    "console_genre", "publisher_genre", "developer_genre",
]
num_cols = [
    "release_year", "release_month", "decade", "year_amount",
    "critic_score", "title_len", "has_number", "title_freq",
]

X_raw = df_lgb[cat_cols + num_cols].copy()
y = np.log1p(df_lgb["total_sales"])
title_text = df_lgb["title"].fillna("")

X_train_raw, X_test_raw, y_train, y_test, title_train, title_test = train_test_split(
    X_raw, y, title_text, test_size=0.2, random_state=42
)

# Target Encoding（学習データで fit → テストへ transform）
te = TargetEncoder(cols=cat_cols, smoothing=10)
X_train_te = X_train_raw.copy()
X_test_te = X_test_raw.copy()
X_train_te[cat_cols] = te.fit_transform(X_train_raw[cat_cols], y_train)
X_test_te[cat_cols] = te.transform(X_test_raw[cat_cols])

# TF-IDF（学習データで fit → テストへ transform）
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), stop_words="english")
X_train_title = tfidf.fit_transform(title_train)
X_test_title = tfidf.transform(title_test)

# 数値特徴量とTF-IDFを結合
X_train = hstack([csr_matrix(X_train_te.astype(np.float32)), X_train_title])
X_test = hstack([csr_matrix(X_test_te.astype(np.float32)), X_test_title])

model = LGBMRegressor(
    n_estimators=5000, learning_rate=0.01,
    max_depth=10, num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.7, random_state=42, verbose=-1,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("R²:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

In [ ]:
# 学習データ / テストデータの R² を比較（過学習の確認）
print("train R²:", r2_score(y_train, model.predict(X_train)))
print("test  R²:", r2_score(y_test, model.predict(X_test)))

## 12. まとめ

| ステップ | 主な施策 |
|---|---|
| ベースライン | One-Hot ＋ 線形回帰（当てはまり不十分） |
| 改善 1 | LightGBM ＋ 目的変数の対数変換 |
| 改善 2 | 2000 年以降に限定（生存バイアス除去）＋ `year_amount` 追加で約 0.3 改善 |
| 改善 3 | タイトルの TF-IDF を追加 |
| 改善 4 | Target Encoding ＋ ハイパーパラメータ調整 |

**最終結果：テスト R² ≈ 0.65**

### 今後の改善余地
- 交差検定（K-Fold）での評価、`early_stopping` による木の本数最適化
- Target Encoding のリーク対策を K-Fold ベースに強化
- 数値特徴量の欠損（`critic_score` 等）の扱いの再検討、欠損フラグの追加
- Optuna 等によるハイパーパラメータ探索
- 予測値・残差の誤差分析（どのジャンル／コンソールで外すか）